!pip install numpy==1.23.5 scikit-learn==1.2.2

In [ ]:
from google.colab import files
files.upload()

In [ ]:
from google.colab import files
files.upload()

In [ ]:
from google.colab import files
files.upload()

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import joblib

model = joblib.load("model.pkl")
vectorizer = joblib.load("vectorizer.pkl")

print("All files loaded successfully ")

In [ ]:
print("Pipeline Stages:")
stages = ["Ingest", "Anonymise", "Embed", "FAISS", "Groq Insights", "Export"]

for i, stage in enumerate(stages, 1):
    print(f"Step {i}: {stage}")

In [ ]:
import pandas as pd

df = pd.read_excel("balanced_edufeed_dataset.xlsx")

print("Total rows:", len(df))
df.head()

In [ ]:
print(df.isnull().sum())

In [ ]:
print(df.columns)

In [ ]:
import re
import pandas as pd

def contains_sensitive(text):
    if pd.isna(text):
        return False

    email = re.search(r'\S+@\S+', str(text))
    numbers = re.search(r'\d{10}', str(text))

    return bool(email or numbers)

In [ ]:
df["sensitive_flag"] = df["comments"].apply(contains_sensitive)

In [ ]:
col = df.columns[0]  # or manually set correct name

df["sensitive_flag"] = df[col].apply(contains_sensitive)

print("Sensitive data found:", df["sensitive_flag"].sum())

In [ ]:
!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

model_embed = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
texts = df["comments"].astype(str).tolist()

In [ ]:
import time

start = time.time()

embeddings = model_embed.encode(texts[:500])  # use sample first

end = time.time()

print("Embedding time:", end - start)

In [ ]:
!pip install faiss-cpu

In [ ]:
import faiss
import numpy as np

dimension = len(embeddings[0])

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("FAISS index created with", index.ntotal, "vectors")

In [ ]:
query = embeddings[0].reshape(1, -1)

start = time.time()
D, I = index.search(query, k=5)
end = time.time()

print("Search time:", end - start)

In [ ]:
import json

results = {
    "sample_results": I.tolist(),
    "distances": D.tolist()
}

with open("results.json", "w") as f:
    json.dump(results, f)

print("Results exported")

In [ ]:
!ls

In [ ]:
import json

with open("results.json", "r") as f:
    data = json.load(f)

print("Export verified ")